In [1]:
import re

# --- 1. Unsloth base (your existing working cell) ---
import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
%pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
%pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
%pip install torchcodec

import torch; torch._dynamo.config.recompile_limit = 64
%pip install --no-deps --upgrade timm

# --- 2. Optimum + ONNX Runtime (NEW) ---
# The CPU and GPU onnxruntime packages CANNOT coexist. Uninstall the CPU one first.
%pip uninstall -y onnxruntime

# Install the ONNX library and ONNX Runtime GPU.
# onnxruntime-gpu 1.19+ defaults to CUDA 12.x / cuDNN 9.x, which matches PyTorch 2.4+ / 2.10.
%pip install onnx onnxruntime-gpu

# Install Optimum base + the ONNX namespace package WITHOUT dependencies.
# --no-deps prevents optimum from upgrading your pinned transformers/torch/etc.
# optimum>=2.1.0 is required for transformers 5.x support.
%pip install --no-deps "optimum>=2.1.0" optimum-onnx

In [2]:
from unsloth import FastModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import torch
import transformers.modeling_utils

# --- Monkey-patch symbols that Optimum still expects but Transformers 5.x removed ---
if not hasattr(transformers.modeling_utils, "get_parameter_dtype"):
    def get_parameter_dtype(parameter):
        try:
            return next(parameter.parameters()).dtype
        except StopIteration:
            # Fallback for modules without parameters (e.g. DataParallel wrappers)
            for m in parameter.modules():
                for p in m.parameters(recurse=False):
                    return p.dtype
                for b in m.buffers(recurse=False):
                    return b.dtype
            return torch.float32
    transformers.modeling_utils.get_parameter_dtype = get_parameter_dtype

if not hasattr(transformers.modeling_utils, "get_parameter_device"):
    def get_parameter_device(parameter):
        try:
            return next(parameter.parameters()).device
        except StopIteration:
            for m in parameter.modules():
                for p in m.parameters(recurse=False):
                    return p.device
                for b in m.buffers(recurse=False):
                    return b.device
            return torch.device("cpu")
    transformers.modeling_utils.get_parameter_device = get_parameter_device

# --- Now the Optimum ONNX import will work ---
from optimum.onnxruntime import ORTModelForCausalLM
print("ORTModelForCausalLM imported successfully")

ImportError: cannot import name '_CAN_RECORD_REGISTRY' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

In [3]:
from optimum.onnxruntime import ORTModelForCausalLM

`Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


ImportError: cannot import name 'get_parameter_dtype' from 'transformers.modeling_utils' (/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py)

In [3]:
from unsloth import FastModel

BASE_MODEL_NAME = 'unsloth/gemma-4-E2B-it'
LORA_MODEL_NAME = 'pameydorke/redred-gemma-4-E2B-it-lora'

model, tokenizer = FastModel.from_pretrained(
    model_name = LORA_MODEL_NAME,
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

ValueError: The checkpoint you are trying to load has model type `gemma4` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [ ]:
import os

merged_path = "/content/merged_model"
os.makedirs(merged_path, exist_ok=True)

# This merges the LoRA into the base weights and saves as a standard HF model
# "merged_16bit" = FP16 full weights (required for clean ONNX export)
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")

config.json:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:32<00:00, 32.90s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:15<00:00, 75.35s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model`


In [ ]:
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

onnx_path = "/content/onnx_model"
os.makedirs(onnx_path, exist_ok=True)

# Export with past-key-values for efficient text generation
ort_model = ORTModelForCausalLM.from_pretrained(
    merged_path,
    export=True,
    task="text-generation-with-past",
    opset=14,  # widely compatible opset
)

tokenizer = AutoTokenizer.from_pretrained(merged_path)

ort_model.save_pretrained(onnx_path)
tokenizer.save_pretrained(onnx_path)

ImportError: cannot import name 'FLAX_WEIGHTS_NAME' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)